In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter
import re

# Constants
MAX_LENGTH = 100
EMBEDDING_DIM = 300
HIDDEN_DIM = 256
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
FINE_TUNE_EPOCHS = 3
GLOVE_PATH = 'dolma_300_2024_1.2M.100_combined.txt'  # Path to GloVe embeddings

# Simple tokenizer function without NLTK dependency
def simple_tokenize(text):
    # Convert to lowercase and tokenize
    text = text.lower()
    # Remove punctuation and split on whitespace
    tokens = re.findall(r'\b\w+\b', text)
    return tokens

# Load and preprocess data
wine_data = pd.read_csv('https://github.com/dvasiliu/AML/blob/main/Data%20Sets/winemagdata130kv2.csv?raw=true',quoting=2)
wine_data = wine_data[['description', 'points']]
wine_data = wine_data.dropna()

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    wine_data['description'],
    wine_data['points'],
    test_size=0.2,
    random_state=42
)

# Tokenize text and build vocabulary
def tokenize_text(text_series):
    return [simple_tokenize(text) for text in text_series]

X_train_tokens = tokenize_text(X_train)
X_test_tokens = tokenize_text(X_test)

# Build vocabulary from training data
word_counts = Counter()
for tokens in X_train_tokens:
    word_counts.update(tokens)

# Create vocabulary (keeping only words that appear at least min_count times)
min_count = 2
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, count in word_counts.items():
    if count >= min_count:
        vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")

# Convert tokens to indices and pad sequences
def texts_to_sequences(tokenized_texts, vocab):
    return [[vocab.get(word, vocab['<UNK>']) for word in text] for text in tokenized_texts]

def pad_sequences(sequences, max_len, pad_value=0):
    padded_sequences = []
    for seq in sequences:
        if len(seq) < max_len:
            padded_seq = seq + [pad_value] * (max_len - len(seq))
        else:
            padded_seq = seq[:max_len]
        padded_sequences.append(padded_seq)
    return padded_sequences

# Convert tokens to sequences
X_train_seq = texts_to_sequences(X_train_tokens, vocab)
X_test_seq = texts_to_sequences(X_test_tokens, vocab)

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, MAX_LENGTH)
X_test_pad = pad_sequences(X_test_seq, MAX_LENGTH)

# 5. Create PyTorch tensors
X_train_tensor = torch.tensor(X_train_pad, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_pad, dtype=torch.long)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float)

# Load GloVe embeddings
def load_glove_embeddings(glove_file, vocab, embedding_dim=300):
    print("Loading GloVe embeddings...")
    embeddings_index = {}
    with open(glove_file, encoding='utf8') as f:
        for line in f:
            values = line.split(' ')
            word = values[0]
            if word in vocab:
                coefs = np.asarray(values[1:], dtype='float32')
                embeddings_index[word] = coefs

    print(f"Found {len(embeddings_index)} word vectors in vocabulary.")

    # Create embedding matrix
    vocab_size = len(vocab)
    embedding_matrix = np.zeros((vocab_size, embedding_dim))

    for word, i in vocab.items():
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

    return embedding_matrix

# Uncomment to load GloVe embeddings when you have the file
# embedding_matrix = load_glove_embeddings(GLOVE_PATH, vocab)
# Initialize embedding matrix as zeros for now
embedding_matrix = np.zeros((len(vocab), EMBEDDING_DIM))

# Create Dataset class
class WineReviewsDataset(Dataset):
    def __init__(self, reviews, ratings):
        self.reviews = reviews
        self.ratings = ratings

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        return {'review': self.reviews[idx], 'rating': self.ratings[idx]}

# Create datasets
train_dataset = WineReviewsDataset(X_train_tensor, y_train_tensor)
test_dataset = WineReviewsDataset(X_test_tensor, y_test_tensor)

# Create dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Define the model
class WineRatingPredictor(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers,
                 bidirectional, dropout, pad_idx, embedding_matrix=None):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        if embedding_matrix is not None:
            self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float))
            self.embedding.weight.requires_grad = False  # Initially freeze embeddings

        self.rnn = nn.GRU(embedding_dim,
                          hidden_dim,
                          num_layers=n_layers,
                          bidirectional=bidirectional,
                          batch_first=True,
                          dropout=0 if n_layers < 2 else dropout)

        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        embedded = self.embedding(text)
        output, hidden = self.rnn(embedded)

        if self.rnn.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])

        return self.fc(hidden)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = WineRatingPredictor(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=1,  # Regression task
    n_layers=N_LAYERS,
    bidirectional=BIDIRECTIONAL,
    dropout=DROPOUT,
    pad_idx=vocab['<PAD>'],
    embedding_matrix=embedding_matrix
).to(device)

# Define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

# Training function
def train_model(model, train_dataloader, test_dataloader, optimizer, criterion, num_epochs, device):
    best_test_loss = float('inf')

    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0

        for batch in train_dataloader:
            reviews = batch['review'].to(device)
            ratings = batch['rating'].to(device)

            optimizer.zero_grad()
            predictions = model(reviews).squeeze(1)
            loss = criterion(predictions, ratings)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Testing
        model.eval()
        test_loss = 0.0

        with torch.no_grad():
            for batch in test_dataloader:
                reviews = batch['review'].to(device)
                ratings = batch['rating'].to(device)

                predictions = model(reviews).squeeze(1)
                loss = criterion(predictions, ratings)

                test_loss += loss.item()

        train_loss = train_loss / len(train_dataloader)
        test_loss = test_loss / len(test_dataloader)

        print(f'Epoch: {epoch+1}/{num_epochs}')
        print(f'Train Loss: {train_loss:.4f}')
        print(f'Test Loss: {test_loss:.4f}')

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            torch.save(model.state_dict(), 'wine_rating_model.pt')
            print('Model saved!')

        print()

    return model

# Train model with frozen embeddings
print("Training with frozen embeddings...")
model = train_model(model, train_dataloader, test_dataloader, optimizer, criterion, NUM_EPOCHS, device)

# Fine-tune with unfrozen embeddings
print("Fine-tuning with unfrozen embeddings...")
for param in model.embedding.parameters():
    param.requires_grad = True

# Use a smaller learning rate for fine-tuning
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE * 0.1)
model = train_model(model, train_dataloader, test_dataloader, optimizer, criterion, FINE_TUNE_EPOCHS, device)

# Evaluate model
def evaluate_model(model, test_dataloader, device):
    model.eval()
    predictions = []
    actuals = []

    with torch.no_grad():
        for batch in test_dataloader:
            reviews = batch['review'].to(device)
            ratings = batch['rating'].to(device)

            pred = model(reviews).squeeze(1)

            predictions.extend(pred.cpu().numpy())
            actuals.extend(ratings.cpu().numpy())

    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)

    print(f'Test MSE: {mse:.4f}')
    print(f'Test MAE: {mae:.4f}')
    print(f'Test R²: {r2:.4f}')

    return mse, mae, r2

# Load best model and evaluate
model.load_state_dict(torch.load('wine_rating_model.pt'))
evaluate_model(model, test_dataloader, device)

Vocabulary size: 18858
Training with frozen embeddings...
Epoch: 1/10
Train Loss: 249.8083
Test Loss: 9.4030
Model saved!

Epoch: 2/10
Train Loss: 24.5224
Test Loss: 9.4319

Epoch: 3/10
Train Loss: 24.4084
Test Loss: 9.5808

Epoch: 4/10
Train Loss: 24.4941
Test Loss: 9.4924

Epoch: 5/10
Train Loss: 24.3101
Test Loss: 9.6090

Epoch: 6/10
Train Loss: 24.5416
Test Loss: 9.3828
Model saved!

Epoch: 7/10
Train Loss: 24.2415
Test Loss: 9.9636

Epoch: 8/10
Train Loss: 24.2921
Test Loss: 9.3797
Model saved!

Epoch: 9/10
Train Loss: 29.6387
Test Loss: 9.4735

Epoch: 10/10
Train Loss: 24.1987
Test Loss: 9.3769
Model saved!

Fine-tuning with unfrozen embeddings...
Epoch: 1/3
Train Loss: 22.1167
Test Loss: 4.4156
Model saved!

Epoch: 2/3
Train Loss: 19.1215
Test Loss: 4.5060

Epoch: 3/3
Train Loss: 18.4292
Test Loss: 4.0052
Model saved!

Test MSE: 4.0033
Test MAE: 1.5787
Test R²: 0.5730


(4.003333795750179, 1.578720371006406, 0.5729952411790695)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
